# Pakages import

In [13]:
import os
import json
import pandas as pd
import yaml
import requests
from requests.auth import HTTPBasicAuth
from bs4 import BeautifulSoup

import ics

# apollo Scraper

data we want: lessons, teachers, type, room, date


In [2]:
with open("config.yaml","r", encoding="UTF-8") as yf:
    config = yaml.safe_load(yf)
print(config)
username = config['credentials']['user']
password = config['credentials']['pass']
Credentials = HTTPBasicAuth(username, password)

{'credentials': {'user': '241956', 'pass': 'Chleb@2005'}}


In [3]:
group_id = input('input group id: ')

#group_id = '252681' # --- IGNORE ---

Url = f"https://planzajec.uek.krakow.pl/index.php?typ=G&id={group_id}&okres=2"

response = requests.get(Url, auth=Credentials)
response.encoding="utf-8"
print(response.status_code)

page_dom = BeautifulSoup(response.text, "html.parser")
print(type(page_dom))

200
<class 'bs4.BeautifulSoup'>


In [4]:
group = page_dom.select_one('div.grupa').get_text(strip=True)
print(group)

ZICSS1-1211


In [5]:
classes_tag = page_dom.select_one('table')
with open('temp.html', 'w', encoding="UTF-8") as hf:
    hf.write(classes_tag.prettify())
#print(classes_tag)
classes = pd.read_html('temp.html', encoding="UTF-8")[0]
os.remove('temp.html')
#print(classes)

### filter out un-neccisery types and only keep lektorat, czwiczenia and exam

In [6]:
classes = classes.loc[classes['Typ'].isin(['lektorat', 'ćwiczenia', 'egzamin'])]
classes = classes[classes['Sala'] != 'Wybierz swoją grupę językową']

In [7]:
classes[['Day','start time', 'hyphen', 'end time', 'duration']] = classes['Dzień, godzina'].str.split(' ', expand=True)

In [8]:
classes['duration'] = classes['duration'].map(lambda x: x.split('(')[1].split('g')[0])

In [9]:
classes = classes.drop(['Dzień, godzina', 'hyphen'], axis=1)

In [10]:
classes["Sala"] = classes["Sala"].str.replace(r'Win.*', '', regex=True)

In [11]:
if not os.path.exists("schedules"):
    os.makedirs("schedules", exist_ok=True)

In [12]:
classes.to_csv(f"schedules/{group}.csv")

ICS callander file - own - Wrong

In [14]:
if not os.path.exists("schedules"):
    os.makedirs("schedules", exist_ok=True)

In [24]:


def to_ics(hf,FileName,
        start_col ="Start time",
        end_col = "End time",
        typ = "Typ",
        lesson = "Przedmiot",
        Prof = "Nauczyciel",
        loc = "Sala"):

  cal=ics.Calendar()

  for index, row in hf.iterrows():
    event = ics.Event()
    event.name = row[lesson]
    event.begin = row[start_col]
    event.end = row[end_col]

    if loc in hf.columns:
      event.loc = row[loc]

    if typ in hf.columns:
      event.typ = row[typ]

    if Prof in hf.colums:
       event.Prof = row[Prof]

    cal.events.add(event)

to_ics(hf, "UniCal.ics")

with open(FileName, 'w') as f:
    f.writelines(cal)
  

AttributeError: '_io.TextIOWrapper' object has no attribute 'iterrows'

AttributeError: '_io.TextIOWrapper' object has no attribute 'iterrows'